In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_openai import ChatOpenAI

from langchain.chains import ConversationalRetrievalChain

In [10]:
from langchain_groq import ChatGroq
llm_model = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

In [5]:
documents = []

pdf_files = [
    "paper.pdf",
    "RAG Paper.pdf"
]

for file in pdf_files:
    loader = PyPDFLoader(file)
    documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 30


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print("Total chunks:", len(docs))

Total chunks: 135


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [8]:
vectorstore = FAISS.from_documents(
    docs,
    embeddings
)

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":4}
)

In [11]:
qa_chain = ConversationalRetrievalChain.from_llm(
    llm_model,
    retriever,
    return_source_documents=True
)

In [12]:
chat_history = []

query = "What problem does the transformer architecture solve?"

result = qa_chain(
    {"question": query, "chat_history": chat_history}
)

print(result["answer"])

C:\Users\Prath\AppData\Local\Temp\ipykernel_2928\4051520288.py:5: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain(


The Transformer architecture solves the problem of sequential computation in natural language processing (NLP) tasks, particularly in tasks that involve long-range dependencies between input and output positions. 

In traditional sequence-to-sequence models, the number of operations required to relate signals from two arbitrary input or output positions grows linearly or logarithmically with the distance between positions. This makes it difficult to learn dependencies between distant positions.

The Transformer architecture addresses this problem by using self-attention and multi-head attention mechanisms, which allow the model to attend to all positions in the input sequence simultaneously and weigh their importance. This reduces the number of operations required to relate signals from two arbitrary input or output positions to a constant, making it easier to learn long-range dependencies.

In particular, the Transformer architecture solves the problem of:

1. Long-range dependencies:

In [13]:
sources = result["source_documents"]

for doc in sources:
    print(doc.metadata)

{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French t

In [ ]:
chat_history.append((query, result["answer"]))

query = "Why is self-attention important?"

result = qa_chain(
    {"question": query, "chat_history": chat_history}
)

print(result["answer"])

Self-attention is important for several reasons:

1. **Efficient Computation**: Self-attention layers connect all positions in a sequence with a constant number of sequentially executed operations, making them faster than recurrent layers when the sequence length is smaller than the representation dimensionality.

2. **Parallelization**: Self-attention allows for more parallelization of computation, as measured by the minimum number of sequential operations required. This is because self-attention can process all positions in a sequence simultaneously, whereas recurrent layers process positions sequentially.

3. **Shorter Path Length**: Self-attention has a shorter path length between long-range dependencies in the network, making it easier to learn such dependencies. This is a key challenge in many sequence transduction tasks.

4. **Ability to Learn Long-Range Dependencies**: Self-attention enables the model to learn long-range dependencies more easily, which is essential for tasks li